# FactIndic 04 – LLM summarisation: Track A, Track B, Track BV

**Track A** – standard prompt. **Track B** – factuality-aware prompt: entity locking, a list of the article's numbers and dates that the summary may use (extracted by code), and chain-of-fact (FACTS → ENTITIES → SUMMARY). **Track BV** – Track B plus verify-and-revise: a rule-based number/date check and an LLM self-check flag unsupported sentences, and the model rewrites the summary once. Same model, length budget and greedy decoding for every track. The notebook then generates questions from every summary (all systems) for the QA-based check in notebook 05. With a 4B model on two T4s, one worker runs on each GPU.

| Setting (right-hand panel) | Value |
|---|---|
| Accelerator | **GPU T4 ×2** |
| Internet | **On** |
| Inputs to attach | output of notebook 03 |
| Time (full profile) | about 1–2 h; the log prints an estimate |

Run it with **Save Version → Save & Run All (Commit)**. When it finishes, its output (folder `factindic_out`) is
ready to attach to the next notebook. Each notebook's output contains every earlier stage, so you only ever attach
the previous notebook.

In [1]:
# ── Setup: make the factindic code importable ─────────────────────────────────────────
# Kaggle: upload factindic-code.zip as a private dataset and attach it
#         (right-hand panel -> Add Input -> Your Work / Datasets -> factindic-code -> "+").
# Colab:  put factindic-code.zip in My Drive (and run the Google Drive cell first), or upload it to /content.
# Or:     push the repository to GitHub and paste its URL here (needs Internet).
REPO_URL = ""   # e.g. "https://github.com/<your-username>/factindic"

import glob, os, shutil, subprocess, sys, zipfile

ON_COLAB = "google.colab" in sys.modules
SRC = "/content/factindic_src" if ON_COLAB else "/kaggle/working/factindic_src"

def _is_pkg(d):
    return all(os.path.exists(os.path.join(d, f)) for f in ("__init__.py", "pipeline.py", "config.py"))

def _candidates():
    inits = glob.glob("/kaggle/input/**/__init__.py", recursive=True)
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    if ON_COLAB:  # shallow search only: Google Drive can be huge
        for base in ("/content", "/content/drive/MyDrive"):
            for depth in ("", "*/", "*/*/"):
                inits += glob.glob(f"{base}/{depth}factindic/__init__.py")
            zips += [z for z in glob.glob(f"{base}/*.zip") + glob.glob(f"{base}/*/*.zip")
                     if "factindic" in os.path.basename(z).lower()]
    return sorted(set(inits)), sorted(set(zips))

def _unzip(z):
    try:
        with zipfile.ZipFile(z) as zf:
            hits = [n for n in zf.namelist() if n.endswith("factindic/__init__.py")]
            if hits:
                shutil.rmtree(SRC, ignore_errors=True)
                zf.extractall(SRC)
                return os.path.dirname(os.path.dirname(os.path.join(SRC, sorted(hits, key=len)[-1])))
    except zipfile.BadZipFile:
        pass
    return None

def _find_code():
    inits, zips = _candidates()
    # 0) Colab: always unpack the newest zip from Drive, so replacing the zip there updates the code
    if ON_COLAB:
        for z in sorted(zips, key=os.path.getmtime, reverse=True):
            code = _unzip(z)
            if code:
                print("using", z)
                return code
    # 1) the repository folder (…/factindic/factindic/__init__.py)
    for p in inits:
        d = os.path.dirname(p)
        if _is_pkg(d) and os.path.basename(d) == "factindic":
            return os.path.dirname(d)
    # 2) only the inner package was uploaded (files directly in the dataset): copy it under the right name
    for p in inits:
        d = os.path.dirname(p)
        if _is_pkg(d):
            shutil.copytree(d, os.path.join(SRC, "factindic"), dirs_exist_ok=True)
            for req in glob.glob(os.path.join(os.path.dirname(d), "**", "requirements.txt"), recursive=True)[:1]:
                shutil.copy(req, SRC)
            return SRC
    # 3) a zip that nobody unpacked
    for z in zips:
        code = _unzip(z)
        if code:
            return code
    # 4) running locally from the repository
    for p in ("factindic", "../factindic"):
        if _is_pkg(p):
            return os.path.abspath(os.path.dirname(p) or ".")
    return None

CODE = _find_code()
if CODE is None and REPO_URL:
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, SRC], check=True)
    CODE = SRC
if CODE is None:
    where = (sorted(glob.glob("/content/*")) + sorted(glob.glob("/content/drive/MyDrive/*"))[:30]) if ON_COLAB else \
            (sorted(glob.glob("/kaggle/input/*")) + sorted(glob.glob("/kaggle/input/*/*"))[:30])
    how = ("Put factindic-code.zip in My Drive and run the Google Drive cell first (or upload the zip to /content)."
           if ON_COLAB else
           "Attach the dataset: right-hand panel -> Add Input -> Your Work (or Datasets) -> factindic-code -> '+'. "
           "If you uploaded it a minute ago, wait until Kaggle finishes processing it.")
    raise FileNotFoundError("factindic code not found.\n" + how + "\nWhat this notebook can see:\n  "
                            + ("\n  ".join(where) if where else "(nothing)"))
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
req = os.path.join(CODE, "requirements.txt")
pkgs = ["-r", req] if os.path.exists(req) else ["indic-transliteration", "wordfreq", "datasketch", "bert-score",
                                                "python-docx", "feedparser", "trafilatura", "peft", "openpyxl",
                                                "bitsandbytes"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
import factindic
from factindic import Config
from factindic import pipeline as P
print("factindic", factindic.__version__, "loaded from", CODE)

factindic 1.0.0 loaded from /kaggle/input/datasets/lkr1196/factindic-code/factindic


In [2]:
# ── Configuration ────────────────────────────────────────────────────────────────────
# Use the SAME profile in every notebook of the chain.
#   "full"  – the real experiment
#   "quick" – ~25 articles per language, to check the whole chain on Kaggle first
#   "mock"  – CPU dry run with stand-in models (no GPU, no downloads); never report its numbers
PROFILE = "full"
cfg = Config.make(PROFILE)
# cfg.summarizer_model = "Qwen/Qwen3-8B"   # bigger model: slower (one process over both GPUs)
# Optional: an OpenAI-compatible API instead of a local model (key stored as a Kaggle secret OPENAI_API_KEY)
# from kaggle_secrets import UserSecretsClient
# os.environ["OPENAI_API_KEY"] = UserSecretsClient().get_secret("OPENAI_API_KEY")
# cfg.backend, cfg.openai_base_url, cfg.openai_model = "openai", "https://api.groq.com/openai/v1", "llama-3.3-70b-versatile"
print(cfg.to_json())

{
  "profile": "full",
  "seed": 13,
  "langs": [
    "hi",
    "bn"
  ],
  "variants": [
    "mono",
    "cm",
    "roman"
  ],
  "data_source": "xlsum",
  "n_test": 200,
  "n_dev": 30,
  "n_train_cm": 60,
  "min_words": 120,
  "max_words_raw": 1200,
  "lead_words": 300,
  "max_sents": 14,
  "min_summary_words": 8,
  "max_summary_words": 60,
  "near_dup_threshold": 0.8,
  "natural_split": [
    0.7,
    0.1,
    0.2
  ],
  "scrape": true,
  "n_natural_per_lang": 100,
  "backend": "auto",
  "summarizer_model": "Qwen/Qwen3-4B-Instruct-2507",
  "codemix_model": "Qwen/Qwen3-4B-Instruct-2507",
  "openai_base_url": "",
  "openai_model": "",
  "openai_concurrency": 4,
  "max_model_len": 6144,
  "max_batch_tokens": 24000,
  "max_batch_size": 24,
  "repetition_penalty": 1.0,
  "workers": -1,
  "quantize": "auto",
  "summary_max_words": 40,
  "tracks": [
    "A",
    "B",
    "BV"
  ],
  "max_new_tokens_A": 448,
  "max_new_tokens_B": 1536,
  "seq2seq_models": [
    "mt5",
    "indicbart"
  ],
 

Optional speed-up: vLLM is several times faster than plain transformers, but installing it replaces Kaggle's PyTorch and can break other packages. Only switch it on if you have time to debug; the default (transformers) works out of the box.

In [3]:
USE_VLLM = False
if USE_VLLM:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], check=True)
    cfg.backend = "vllm"

In [4]:
llm = P.run_stage04(cfg)

Stages available: 01_data, 02_codemix, 03_seq2seq
Summarising 1333 inputs x 2 tracks = 2666 prompts


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Starting 2 worker(s) for 2666 prompts with Qwen/Qwen3-4B-Instruct-2507 (logs in /kaggle/working/factindic_out/_jobs/summarize_8b8fb53b)
  [worker 0] Loading Qwen/Qwen3-4B-Instruct-2507 for 1333 prompts
  [worker 1] Loading Qwen/Qwen3-4B-Instruct-2507 for 1333 prompts
  [worker 1] Loading weights: 100%|██████████| 398/398 [00:08<00:00, 47.28it/s]
  [worker 0] worker 0: 24/1333 generated (1.7 min, ~95 min left; avg batch 12.0, 37 tokens/s)
  [worker 1] Loading weights: 100%|██████████| 398/398 [00:08<00:00, 47.28it/s]
  [worker 0] worker 0: 24/1333 generated (1.7 min, ~95 min left; avg batch 12.0, 37 tokens/s)
  [worker 1] Loading weights: 100%|██████████| 398/398 [00:08<00:00, 47.28it/s]
  [worker 0] worker 0: 24/1333 generated (1.7 min, ~95 min left; avg batch 12.0, 37 tokens/s)
  [worker 1] Loading weights: 100%|██████████| 398/398 [00:08<00:00, 47.28it/s]
  [worker 0] worker 0: 24/1333 generated (1.7 min, ~95 min left; avg batch 12.0, 37 tokens/s)
  [worker 1] Loading weights: 100%|█

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

... (4 outputs left out)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Starting 2 worker(s) for 3677 prompts with Qwen/Qwen3-4B-Instruct-2507 (logs in /kaggle/working/factindic_out/_jobs/qg_f107b54d)
  [worker 0] Loading Qwen/Qwen3-4B-Instruct-2507 for 1839 prompts
  [worker 1] Loading Qwen/Qwen3-4B-Instruct-2507 for 1838 prompts
  [worker 1] Loading weights: 100%|██████████| 398/398 [00:08<00:00, 46.83it/s]
  [worker 0] worker 0: 24/1839 generated (0.9 min, ~67 min left; avg batch 24.0, 120 tokens/s)
  [worker 1] worker 1: 24/1838 generated (1.0 min, ~76 min left; avg batch 24.0, 104 tokens/s)
  [worker 0] worker 0: 24/1839 generated (0.9 min, ~67 min left; avg batch 24.0, 120 tokens/s)
  [worker 1] worker 1: 24/1838 generated (1.0 min, ~76 min left; avg batch 24.0, 104 tokens/s)
  [worker 0] worker 0: 24/1839 generated (0.9 min, ~67 min left; avg batch 24.0, 120 tokens/s)
  [worker 1] worker 1: 24/1838 generated (1.0 min, ~76 min left; avg batch 24.0, 104 tokens/s)
  [worker 0] worker 0: 120/1839 generated (4.2 min, ~60 min left; avg batch 24.0, 131 tok

,lang,track,variant,n,time_s,words,errors,cut_pct,empty_pct
0,bn,A,cm,200,4.703885,23.560000,0,0.5,0.0
1,bn,A,mono,200,4.871916,22.140000,0,0.0,0.0
2,bn,A,natural,61,4.697205,22.409836,0,0.0,0.0
3,bn,A,roman,200,3.177479,24.360000,0,2.5,0.0
4,bn,B,cm,200,18.882470,28.920000,0,0.0,1.5
5,bn,B,mono,200,20.741849,26.985000,0,0.0,0.0
6,bn,B,natural,61,19.839685,25.557377,0,0.0,0.0
7,bn,B,roman,200,13.939628,34.190000,0,1.5,2.0
8,bn,BV,cm,200,26.661662,26.455000,0,0.5,0.0
9,bn,BV,mono,200,27.943735,25.750000,0,0.0,0.0


In [5]:
from IPython.display import Markdown, display
display(Markdown(open(f"{cfg.out_root}/04_llm/llm_report.md", encoding="utf-8").read()))

# LLM summaries

| lang | track | variant | n | time_s | words | errors | cut_pct | empty_pct |
|---|---|---|---|---|---|---|---|---|
| bn | A | cm | 200 | 4.70 | 23.56 | 0 | 0.50 | 0.00 |
| bn | A | mono | 200 | 4.87 | 22.14 | 0 | 0.00 | 0.00 |
| bn | A | natural | 61 | 4.70 | 22.41 | 0 | 0.00 | 0.00 |
| bn | A | roman | 200 | 3.18 | 24.36 | 0 | 2.50 | 0.00 |
| bn | B | cm | 200 | 18.88 | 28.92 | 0 | 0.00 | 1.50 |
| bn | B | mono | 200 | 20.74 | 26.98 | 0 | 0.00 | 0.00 |
| bn | B | natural | 61 | 19.84 | 25.56 | 0 | 0.00 | 0.00 |
| bn | B | roman | 200 | 13.94 | 34.19 | 0 | 1.50 | 2.00 |
| bn | BV | cm | 200 | 26.66 | 26.45 | 0 | 0.50 | 0.00 |
| bn | BV | mono | 200 | 27.94 | 25.75 | 0 | 0.00 | 0.00 |
| bn | BV | natural | 61 | 26.12 | 25.34 | 0 | 0.00 | 0.00 |
| bn | BV | roman | 200 | 19.71 | 27.45 | 0 | 1.50 | 0.00 |
| hi | A | cm | 200 | 4.05 | 33.49 | 0 | 0.50 | 0.00 |
| hi | A | mono | 200 | 4.13 | 33.12 | 0 | 0.00 | 0.00 |
| hi | A | natural | 72 | 4.49 | 31.82 | 0 | 0.00 | 0.00 |
| hi | A | roman | 200 | 3.11 | 34.38 | 0 | 1.50 | 0.00 |
| hi | B | cm | 200 | 13.17 | 36.26 | 0 | 0.00 | 0.00 |
| hi | B | mono | 200 | 16.91 | 35.66 | 0 | 0.00 | 0.00 |
| hi | B | natural | 72 | 17.26 | 34.25 | 0 | 0.00 | 0.00 |
| hi | B | roman | 200 | 7.87 | 42.19 | 0 | 0.50 | 0.00 |
| hi | BV | cm | 200 | 19.52 | 34.54 | 0 | 0.00 | 0.00 |
| hi | BV | mono | 200 | 22.68 | 34.31 | 0 | 0.00 | 0.00 |
| hi | BV | natural | 72 | 22.14 | 33.26 | 0 | 0.00 | 0.00 |
| hi | BV | roman | 200 | 12.70 | 36.62 | 0 | 1.00 | 0.00 |

## Track BV: share of B summaries flagged

| lang | variant | BV flagged share |
|---|---|---|
| bn | cm | 0.65 |
| bn | mono | 0.54 |
| bn | natural | 0.4426229508196721 |
| bn | roman | 0.81 |
| hi | cm | 0.59 |
| hi | mono | 0.475 |
| hi | natural | 0.3611111111111111 |
| hi | roman | 0.695 |

Questions generated for 3677 summaries.


Same article, three inputs, three tracks:

In [6]:
one = llm[llm.role == "test"].doc_id.iloc[0]
for r in llm[llm.doc_id == one].sort_values(["variant", "track"]).itertuples():
    print(f"[{r.variant:5s} {r.track:2s}] {r.summary}")

[cm    A ] उपराष्ट्रपति शेखावत ने बर्मा की यात्रा करके दोनों देशों के बीच शिक्षा, अनुसंधान और विज्ञान के साझा कार्यक्रमों के साथ एक समझौता किया। इसके साथ ही दोनों देशों के नागरिकों को वीजा छूट देने का समझौता हुआ।
[cm    B ] दिवंगत प्रधानमंत्री राजीव गाँधी 1987 में बर्मा गए थे। उपराष्ट्रपति शेखावत बर्मा के अधिकारियों से diplomacy और regional issues पर बातचीत करेंगे और education और research के क्षेत्र में joint programs के लिए deal हुआ।
[cm    BV] दिवंगत प्रधानमंत्री राजीव गाँधी 1987 में बर्मा गए थे। उपराष्ट्रपति शेखावत बर्मा के अधिकारियों से diplomacy और regional issues पर बातचीत करेंगे।
[mono  A ] उपराष्ट्रपति शेखावत बर्मा की यात्रा कर रहे हैं, जिसमें द्विपक्षीय समझौते और वीज़ा छूट के साथ राजनीतिक, व्यापार और सांस्कृतिक संबंधों में बेहतरी का उद्देश्य है।
[mono  B ] दिवंगत प्रधानमंत्री राजीव गाँधी 1987 में बर्मा गए थे। उपराष्ट्रपति शेखावत बर्मा के अधिकारियों से द्विपक्षीय और क्षेत्रीय मामलों पर बातचीत करेंगे और शिक्षा और अनुसंधान में संयुक्त कार्यक्रम शुरू करने के समझौते भी हुए हैं।
[mo

In [7]:
import os
for root, dirs, files in os.walk(cfg.out_root):
    dirs[:] = [d for d in dirs if not d.startswith("_")]
    depth = root[len(cfg.out_root):].count(os.sep)
    if depth <= 1:
        print("  " * depth + os.path.basename(root) + "/", ", ".join(sorted(files))[:300])

factindic_out/ 
  03_seq2seq/ config.json, lora_training.json, summaries_seq2seq.jsonl
  01_data/ config.json, data_report.md, data_stats.csv, docs.jsonl
  02_codemix/ codemix_quality_check.xlsx, codemix_report.md, config.json, env.json, variants.jsonl
  04_llm/ config.json, env.json, llm_report.md, qg.jsonl, summaries_llm.jsonl
